# 01b — Clean the dataset labels (Release v1.0 → v1.1)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarEAbdelaal/M4U3-CV-Facade-Materials/blob/main/notebooks/01b_clean_dataset.ipynb)

Turns the frozen Roboflow export (Release `v1.0`, Roboflow version 4) into a cleaned dataset (Release `v1.1`). Images are not changed; only the label files are rewritten, and some images are removed.

**Why.** Inspection found (see `docs/error_analysis.md`, A1–A6): mask outlines mixed with boxes, thousands of tiny fragments, the same material labelled per pane / per stone in some classes and per wall in others, boxes nested inside boxes of the same class, two different materials stacked on the same area, and images missing a label for their own material.

**Rules, applied to every label file in order**

| Step | Rule | Setting |
|---|---|---|
| 1 | Polygon (SAM mask) → its tightest box | — |
| 2 | Drop fragments narrower or shorter than 4 px at 640 px | `MIN_BOX_PX` |
| 3 | Same-class boxes that overlap or are closer than 0.5% of the image width → one box per continuous area (all classes). This also removes nested duplicates | `MERGE_GAP` |
| 4 | Two different classes on (almost) the same area (IoU > 0.7) → keep the class the photo was collected for, otherwise the larger box | `STACK_IOU` |
| 5 | Drop boxes smaller than 1% of the image area | `MIN_AREA` |
| 6 | Remove images left with no labels, or with no label of their own material (listed for relabelling) | `DROP_IMAGES_MISSING_OWN_CLASS` |

The output zip is written deterministically (fixed order, fixed timestamps, no compression), so running this notebook again gives a file with the same SHA256.

No credentials needed; no GPU needed.

In [ ]:
# Cell 1 — settings
from pathlib import Path

MIN_BOX_PX = 4            # step 2
MERGE_GAP = 0.005         # step 3: 0.5% of image width (about 3 px at 640)
STACK_IOU = 0.7           # step 4
MIN_AREA = 0.01           # step 5: 1% of the image area (about 64 x 64 px at 640)
DROP_IMAGES_MISSING_OWN_CLASS = True   # step 6

OUT_NAME = "facade-materials-v4-clean-yolov8.zip"
CLEAN = Path("/content/dataset_clean")
REPORT = Path("/content/cleaning_report")
REPORT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Cell 2 — download the frozen v1.0 dataset from the GitHub Release (course standard cell)
import os, hashlib, zipfile, urllib.request, yaml
from pathlib import Path

DATA_URL = "https://github.com/OmarEAbdelaal/M4U3-CV-Facade-Materials/releases/download/v1.0/facade-materials-v3-yolov8.zip"
SHA256 = "134e41be8c310452bbe8ea33fd55a891c6e29ca863f3c9b8df3827f59bb951cd"
ROOT = Path("/content/dataset")
ZIP_PATH = Path("/content/dataset.zip")

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def find_data_yaml(root: Path) -> Path:
    hits = list(root.rglob("data.yaml"))
    if not hits:
        raise FileNotFoundError(f"No data.yaml under {root}")
    return hits[0]

already_ready = (ROOT / "data.yaml").exists() or any(ROOT.glob("*/data.yaml"))
if not already_ready:
    ROOT.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    digest = sha256_file(ZIP_PATH)
    if digest != SHA256:
        ZIP_PATH.unlink(missing_ok=True)
        raise AssertionError(f"Checksum mismatch: {digest}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(ROOT)

cfg_path = find_data_yaml(ROOT)
dataset_root = cfg_path.parent
cfg = yaml.safe_load(cfg_path.read_text())
names = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][k] for k in sorted(cfg["names"])]
print(names)

In [ ]:
# Cell 3 — cleaning functions
import collections, re
import numpy as np, cv2

def read_boxes(lines, min_norm):
    """Steps 1–2: polygons to boxes, drop fragments. Returns [cls, cx, cy, w, h]."""
    boxes = []
    for line in lines:
        p = line.split()
        if not p:
            continue
        if len(p) > 5:
            xs, ys = [float(v) for v in p[1::2]], [float(v) for v in p[2::2]]
            x1, x2 = max(0.0, min(xs)), min(1.0, max(xs))
            y1, y2 = max(0.0, min(ys)), min(1.0, max(ys))
            p = [p[0], (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]
        if len(p) != 5:
            continue
        c, cx, cy, w, h = int(p[0]), *map(float, p[1:])
        if min(w, h) >= min_norm:
            boxes.append([c, cx, cy, w, h])
    return boxes

def merge_areas(boxes, gap, grid=640):
    """Step 3: same-class boxes that overlap or nearly touch become one box (union of the originals)."""
    out = []
    g = max(0, int(round(gap * grid)))
    for c in sorted({b[0] for b in boxes}):
        corners = [(cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2) for k, cx, cy, w, h in boxes if k == c]
        mask = np.zeros((grid, grid), np.uint8)
        for x1, y1, x2, y2 in corners:
            mask[max(0, int(y1 * grid) - g):min(grid, int(np.ceil(y2 * grid)) + g),
                 max(0, int(x1 * grid) - g):min(grid, int(np.ceil(x2 * grid)) + g)] = 1
        _, lab = cv2.connectedComponents(mask, connectivity=8)
        groups = collections.defaultdict(list)
        for x1, y1, x2, y2 in corners:
            px = min(grid - 1, max(0, int((x1 + x2) / 2 * grid)))
            py = min(grid - 1, max(0, int((y1 + y2) / 2 * grid)))
            groups[int(lab[py, px])].append((x1, y1, x2, y2))
        for m in groups.values():
            x1, y1 = max(0.0, min(a[0] for a in m)), max(0.0, min(a[1] for a in m))
            x2, y2 = min(1.0, max(a[2] for a in m)), min(1.0, max(a[3] for a in m))
            out.append([c, (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1])
    return out

def _xyxy(b):
    return (b[1] - b[3] / 2, b[2] - b[4] / 2, b[1] + b[3] / 2, b[2] + b[4] / 2)

def _iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0

def resolve_stacked(boxes, own_class):
    """Step 4: keep one class where two different classes cover (almost) the same area."""
    drop, log = set(), []
    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            if i in drop or j in drop or boxes[i][0] == boxes[j][0]:
                continue
            if _iou(_xyxy(boxes[i]), _xyxy(boxes[j])) > STACK_IOU:
                ci, cj = names[boxes[i][0]], names[boxes[j][0]]
                if ci == own_class:
                    loser = j
                elif cj == own_class:
                    loser = i
                else:
                    loser = i if boxes[i][3] * boxes[i][4] < boxes[j][3] * boxes[j][4] else j
                drop.add(loser)
                log.append((names[boxes[loser][0]], names[boxes[i + j - loser][0]]))
    return [b for k, b in enumerate(boxes) if k not in drop], log

def clean_file(lines, own_class):
    b = read_boxes(lines, MIN_BOX_PX / 640)
    b = merge_areas(b, MERGE_GAP)
    b, log = resolve_stacked(b, own_class)
    small = [x for x in b if x[3] * x[4] < MIN_AREA]
    b = [x for x in b if x[3] * x[4] >= MIN_AREA]
    b.sort(key=lambda x: (x[0], round(x[1], 6), round(x[2], 6)))
    return b, small, log

def source_of(stem):
    """'brick_008_jpg.rf.<hash>' -> 'brick_008'; the folder class is the part before the number."""
    return re.sub(r"_(jpg|jpeg|png)\.rf\..*$", "", stem)

def own_class_of(src):
    return src.rsplit("_", 1)[0]

In [ ]:
# Cell 4 — clean every label file and write the cleaned dataset
import shutil, csv

if CLEAN.exists():
    shutil.rmtree(CLEAN)

stats = collections.Counter()
before, after = collections.Counter(), collections.Counter()
stacked_log, dropped = [], {}
cleaned = {}          # (split, stem) -> boxes

for split in ["train", "valid", "test"]:
    for lbl in sorted((dataset_root / split / "labels").glob("*.txt")):
        src = source_of(lbl.stem)
        own = own_class_of(src)
        lines = lbl.read_text().splitlines()
        for b in read_boxes(lines, MIN_BOX_PX / 640):
            before[(split, names[b[0]])] += 1
        boxes, small, log = clean_file(lines, own)
        stats["small_dropped"] += len(small)
        stacked_log += [(split, lbl.stem, loser, winner) for loser, winner in log]
        if not boxes:
            dropped[src] = "no labels left"
        elif DROP_IMAGES_MISSING_OWN_CLASS and own in names and own not in {names[b[0]] for b in boxes}:
            dropped.setdefault(src, f"no {own} label")
        cleaned[(split, lbl.stem)] = boxes

for (split, stem), boxes in cleaned.items():
    if source_of(stem) in dropped:
        stats[f"{split}_images_removed"] += 1
        continue
    for sub in ["images", "labels"]:
        (CLEAN / split / sub).mkdir(parents=True, exist_ok=True)
    img = next((dataset_root / split / "images").glob(stem + ".*"))
    shutil.copy2(img, CLEAN / split / "images" / img.name)
    (CLEAN / split / "labels" / f"{stem}.txt").write_text(
        "".join(f"{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n" for b in boxes))
    for b in boxes:
        after[(split, names[b[0]])] += 1
    stats[f"{split}_images_kept"] += 1

clean_cfg = {"path": ".", "train": "train/images", "val": "valid/images", "test": "test/images",
             "nc": len(names), "names": names,
             "source": {"release": "v1.0", "sha256": SHA256, "roboflow": cfg.get("roboflow", {})},
             "cleaning": {"min_box_px": MIN_BOX_PX, "merge_gap": MERGE_GAP, "stack_iou": STACK_IOU,
                          "min_area": MIN_AREA, "drop_images_missing_own_class": DROP_IMAGES_MISSING_OWN_CLASS,
                          "notebook": "notebooks/01b_clean_dataset.ipynb"}}
(CLEAN / "data.yaml").write_text(yaml.safe_dump(clean_cfg, sort_keys=False))
for extra in ["README.dataset.txt", "README.roboflow.txt"]:
    if (dataset_root / extra).exists():
        shutil.copy2(dataset_root / extra, CLEAN / extra)

with open(REPORT / "removed_images.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["source_image", "reason"]); w.writerows(sorted(dropped.items()))
with open(REPORT / "stacked_labels_resolved.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["split", "label_file", "removed_class", "kept_class"]); w.writerows(stacked_log)
print(dict(stats))
print(f"{len(dropped)} source images removed; {len(stacked_log)} stacked labels resolved")

In [ ]:
# Cell 5 — boxes per class and split, before and after
import pandas as pd

rows = [{"class": c, "split": s, "before": before[(s, c)], "after": after[(s, c)]}
        for s in ["train", "valid", "test"] for c in names]
summary = pd.DataFrame(rows).pivot(index="class", columns="split", values=["before", "after"])
summary.loc["total"] = summary.sum()
summary.to_csv(REPORT / "boxes_before_after.csv")
summary

In [ ]:
# Cell 6 — before / after on 6 images
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

COLORS = {"brick": (220, 70, 50), "cladding_panel": (40, 150, 255), "exposed_concrete": (150, 150, 150),
          "glass": (0, 220, 220), "painted_render": (255, 170, 0), "stone_cladding": (170, 90, 255)}
SAMPLES = ["exposed_concrete_005", "cladding_panel_003", "painted_render_009", "brick_008", "glass_002", "glass_021"]

def draw(img_path, boxes):
    im = Image.open(img_path).convert("RGB")
    W, H = im.size
    d = ImageDraw.Draw(im)
    for c, cx, cy, w, h in boxes:
        d.rectangle([(cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H],
                    outline=COLORS[names[c]], width=3)
    return im

fig, axes = plt.subplots(len(SAMPLES), 2, figsize=(9, 4.4 * len(SAMPLES)))
for r, src in enumerate(SAMPLES):
    lbl = next((dataset_root / s / "labels" / p.name) for s in ["train", "valid", "test"]
               for p in sorted((dataset_root / s / "labels").glob(src + "_*.txt")))
    img = next(lbl.parent.parent.joinpath("images").glob(lbl.stem + ".*"))
    raw = read_boxes(lbl.read_text().splitlines(), MIN_BOX_PX / 640)
    new, _, _ = clean_file(lbl.read_text().splitlines(), own_class_of(src))
    for ax, boxes, title in ((axes[r, 0], raw, "before"), (axes[r, 1], new, "after")):
        ax.imshow(draw(img, boxes)); ax.set_title(f"{src} {title}: {len(boxes)} boxes", fontsize=9); ax.axis("off")
plt.tight_layout()
plt.savefig(REPORT / "cleaning_before_after.jpg", dpi=90)
plt.show()

In [ ]:
# Cell 7 — write the cleaned zip deterministically and print its SHA256
import zipfile

out_zip = Path("/content") / OUT_NAME
files = sorted(p for p in CLEAN.rglob("*") if p.is_file())
with zipfile.ZipFile(out_zip, "w", compression=zipfile.ZIP_STORED) as z:
    for p in files:
        info = zipfile.ZipInfo(str(p.relative_to(CLEAN)), date_time=(2026, 9, 27, 0, 0, 0))
        info.external_attr = 0o644 << 16
        z.writestr(info, p.read_bytes())
digest = sha256_file(out_zip)
(REPORT / "sha256.txt").write_text(f"{digest}  {OUT_NAME}\n")
print(out_zip, f"{out_zip.stat().st_size:,} bytes")
print("SHA256:", digest)
try:
    from google.colab import files as colab_files
    shutil.make_archive("/content/cleaning_report", "zip", REPORT)
    colab_files.download(str(out_zip))
    colab_files.download("/content/cleaning_report.zip")
except Exception:
    pass